# Partitioned LDSC

Goal: run partitioned LDSC in the refactored codebase by building query annotations, computing baseline-plus-query LD scores from an R2-table reference panel, and regressing one trait on that partitioned design.

Path-token rules used below:
- use `@` for chromosome suites such as `baseline.@`
- use globs when the filenames do not follow the simple chromosome-suffix convention
- scalar inputs still resolve to exactly one file
- output prefixes remain literal destinations


In [ ]:
from pathlib import Path

import sys

# Set this explicitly for your environment.
SRC_DIR = Path("/absolute/path/to/ldsc_py3_restructured/src")

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Set SRC_DIR to an existing path before running the notebook: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

## Process Annotation Files

In [ ]:
from ldsc import CommonConfig, run_bed_to_annot

In [ ]:
# Set these explicitly for your environment.
DATA_DIR = Path("/absolute/path/to/resources")
OUTPUT_DIR = Path("/absolute/path/to/result_consistency_checks/output_new")

BED_FILES = str(DATA_DIR / "annot_raw" / "*.bed")
BASELINE_ANNOT_DIR = DATA_DIR / "baseline_v1.2"
QUERY_ANNOT_OUTPUT_DIR = OUTPUT_DIR / "annot_processed"
COMMON_CONFIG = CommonConfig(snp_identifier="chr_pos", genome_build="hg19")

for path_name, path_value in {
    "DATA_DIR": DATA_DIR,
    "OUTPUT_DIR": OUTPUT_DIR,
    "BASELINE_ANNOT_DIR": BASELINE_ANNOT_DIR,
}.items():
    if not path_value.exists():
        raise FileNotFoundError(f"Set {path_name} to an existing path before running the notebook: {path_value}")

In [ ]:
run_bed_to_annot(
    bed_files=BED_FILES,
    baseline_annot_dir=BASELINE_ANNOT_DIR,
    output_dir=QUERY_ANNOT_OUTPUT_DIR,
)

## Calculate LD Scores

In [ ]:
from ldsc import (
    AnnotationBuildConfig,
    AnnotationBuilder,
    AnnotationSourceSpec,
    CommonConfig,
    LDScoreCalculator,
    LDScoreConfig,
    RefPanelLoader,
    RefPanelSpec,
    RegressionConfig,
    RegressionRunner,
)

In [ ]:
# Reuse DATA_DIR and OUTPUT_DIR from the annotation step above.
COMMON_CONFIG = CommonConfig(snp_identifier="chr_pos", genome_build="hg19")

BASELINE_ANNOT_PATHS = DATA_DIR / "baseline_v1.2" / "baseline.@"
QUERY_ANNOT_PATHS = OUTPUT_DIR / "annot_processed" / "query.@"
R2_TABLE_PATHS = DATA_DIR / "ref_panel" / "MAF0.01_SNPonly_hm3only" / "parquet" / "ld" / "EUR_chr@_LD.parquet"
LD_WIND_CM = 1.0

for path_name, path_value in {
    "BASELINE_ANNOT_DIR": DATA_DIR / "baseline_v1.2",
    "QUERY_ANNOT_OUTPUT_DIR": OUTPUT_DIR / "annot_processed",
}.items():
    if not path_value.exists():
        raise FileNotFoundError(f"Set {path_name} to an existing path before running the notebook: {path_value}")

In [ ]:
annotation_bundle = AnnotationBuilder(COMMON_CONFIG, AnnotationBuildConfig()).run(
    AnnotationSourceSpec(
        baseline_annot_paths=BASELINE_ANNOT_PATHS,
        query_annot_paths=QUERY_ANNOT_PATHS,
    )
)

ref_panel = RefPanelLoader(COMMON_CONFIG).load(
    RefPanelSpec(
        backend="parquet_r2",
        r2_table_paths=R2_TABLE_PATHS,
        chromosomes=tuple(annotation_bundle.chromosomes),
    )
)

ldscore_result = LDScoreCalculator().run(
    annotation_bundle=annotation_bundle,
    ref_panel=ref_panel,
    ldscore_config=LDScoreConfig(ld_wind_cm=LD_WIND_CM),
    common_config=COMMON_CONFIG,
)

runner = RegressionRunner(COMMON_CONFIG, RegressionConfig())

Since the annotation files and reference panel do not cover exactly the same SNP universe, the LD-score run can only use SNPs that exist in both:
- the annotation bundle
- the R2 table

In [ ]:
before = annotation_bundle.metadata["CHR"].value_counts().sort_index()
after = ldscore_result.reference_metadata["CHR"].value_counts().sort_index()

comparison = (
    before.rename("annotation_bundle")
    .to_frame()
    .join(after.rename("retained_in_ldscore"), how="outer")
    .fillna(0)
    .astype(int)
)
comparison["dropped"] = comparison["annotation_bundle"] - comparison["retained_in_ldscore"]
comparison

## Estimate Partitioned Heritability

In [ ]:
from ldsc import load_sumstats

In [ ]:
SUMSTATS_PATH = OUTPUT_DIR / "sumstats_processed" / "mdd2025.sumstats.gz"
TRAIT_NAME = "mdd2025"
PARTITIONED_OUTPUT_PATH = OUTPUT_DIR / "pldsc_results" / "partitioned_h2.tsv"

if not SUMSTATS_PATH.exists():
    raise FileNotFoundError(f"Set SUMSTATS_PATH to an existing path before running the notebook: {SUMSTATS_PATH}")

In [ ]:
sumstats = load_sumstats(
    SUMSTATS_PATH,
    trait_name=TRAIT_NAME,
)

partitioned = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
)

PARTITIONED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
partitioned.to_csv(PARTITIONED_OUTPUT_PATH, sep="\t", index=False)
partitioned